# 3. Benchmarking against ICA

Comparing the trained model against independent component analysis, the standard automated
method for handling ocular artifacts in EEG.

Both methods are run over the same held out recordings and compared with a paired
significance test.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from ocular import ica_baseline, manifest, metrics, model as model_module, splits, stats
from ocular.benchmark import format_summary, run
from ocular.data import LABEL_TO_INDEX
from ocular.evaluate import predict

RAW = Path("../data/raw")
MANIFEST = Path("../data/manifest.csv")
ARTIFACTS = Path("../artifacts")

## Baseline definition

ICA identifies an ocular component rather than labelling individual segments, so three
choices are needed to produce a comparable per segment decision.

**Segment score.** Each segment is scored by the peak to peak swing of the ocular component
within it. A blink is a large, brief deflection, so range separates it more sharply than
variance.

**Standardisation.** ICA components carry an arbitrary sign and scale that varies between
recordings, so scores are standardised within each recording using the median and MAD. A
single raw cutoff across participants would penalise the baseline for reasons unrelated to
class separation.

**Threshold.** Fitted on the validation recordings by maximising balanced accuracy, then
applied unchanged to the test recordings.

The baseline reads the EOG electrodes; the model does not. This asymmetry favours the
baseline and is retained, since the model is intended to work without an EOG montage.

In [ ]:
frame = manifest.read(MANIFEST)
model, metadata = model_module.load(ARTIFACTS / "model.pt")
device = model_module.pick_device()

stored = metadata["split"]
split = splits.Split(train=tuple(stored["train"]), val=tuple(stored["val"]),
                     test=tuple(stored["test"]))
assigned = split.assign(frame)

val_frame = assigned[assigned["split"] == "val"].reset_index(drop=True)
test_frame = assigned[assigned["split"] == "test"].reset_index(drop=True)

print(f"validation: {len(val_frame)} segments, {val_frame['group'].nunique()} recordings")
print(f"test:       {len(test_frame)} segments, {test_frame['group'].nunique()} recordings")
print()
print("test recordings:", sorted(test_frame["group"].unique()))

## Running ICA

ICA is fit per recording on a 1 Hz highpassed copy, since the decomposition is unstable in
the presence of slow drifts. Ocular components are identified with MNE's `find_bads_eog`.
If no component passes its correlation threshold, the most EOG correlated component is used.

This is the slowest step in the pipeline.

In [ ]:
needed = set(val_frame["group"]) | set(test_frame["group"])
ica_scores = ica_baseline.score_dataset(RAW, groups=needed)

print(f"{len(ica_scores)} segments scored across {ica_scores['group'].nunique()} recordings")
ica_scores.head()

In [ ]:
val_scored = val_frame.merge(ica_scores[["key", "ica_score"]], on="key")
test_scored = test_frame.merge(ica_scores[["key", "ica_score"]], on="key")

fig, ax = plt.subplots(figsize=(9, 4))
for label, subset in test_scored.groupby("label"):
    ax.hist(subset["ica_score"], bins=60, alpha=0.6, label=label, density=True)
ax.set_xlabel("standardised ICA component swing")
ax.set_ylabel("density")
ax.set_title("ICA baseline score distribution, test recordings")
ax.legend()
plt.show()

Overlap between the two distributions accounts for the baseline's errors, and consists
mainly of saccades. A large vertical eye movement drives the ocular component to a similar
amplitude as a blink, which a single threshold cannot separate.

In [ ]:
threshold, val_balanced = ica_baseline.calibrate_threshold(
    val_scored["ica_score"].to_numpy(),
    val_scored["label"].map(LABEL_TO_INDEX).to_numpy(),
)
print(f"threshold {threshold:.3f} fitted on validation")
print(f"validation balanced accuracy at that threshold: {val_balanced:.4f}")

## Scoring both methods

Row order is preserved through prediction, so the two sets of decisions align segment by
segment. This pairing is required for the significance test below.

In [ ]:
y_true, model_pred, model_score = predict(model, test_scored, device)
baseline_pred = (test_scored["ica_score"].to_numpy() >= threshold).astype(int)

model_metrics = metrics.compute(y_true, model_pred, model_score)
baseline_metrics = metrics.compute(y_true, baseline_pred, test_scored["ica_score"].to_numpy())

print(metrics.format_report("model", model_metrics))
print()
print(metrics.format_report("ICA baseline", baseline_metrics))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
for ax, (name, scores) in zip(axes, [("Model", model_metrics), ("ICA baseline", baseline_metrics)]):
    cm = np.array(scores["confusion_matrix"], dtype=float)
    normalised = cm / cm.sum(axis=1, keepdims=True)
    ax.imshow(normalised, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks([0, 1], ["non-blink", "blink"])
    ax.set_yticks([0, 1], ["non-blink", "blink"])
    ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title(name)
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f"{int(cm[i, j])}\n{normalised[i, j]:.1%}", ha="center",
                    va="center", color="white" if normalised[i, j] > 0.5 else "black")
plt.tight_layout()
plt.show()

## Accuracy by event type

Rest, horizontal saccades and vertical saccades all belong to the negative class, but they
differ in difficulty. Breaking the results down by event type shows where each method fails.

In [ ]:
rows = []
for event in ("rest", "h_saccade", "v_saccade", "blink"):
    mask = test_scored["event"].to_numpy() == event
    if not mask.sum():
        continue
    rows.append({
        "event": event,
        "n": int(mask.sum()),
        "model": float(np.mean(model_pred[mask] == y_true[mask])),
        "ica": float(np.mean(baseline_pred[mask] == y_true[mask])),
    })

pd.DataFrame(rows).set_index("event").round(3)

## McNemar's test

Both methods are scored on the same segments, so their errors are paired and an independent
two sample test does not apply.

McNemar's test uses only the segments where the two methods disagree. Under the null
hypothesis that both methods are equally accurate, each should be correct on half of those
segments, and the test measures how improbable the observed imbalance is.

The exact binomial form is used below 25 disagreements, where the chi squared approximation
is unreliable.

In [ ]:
comparison = stats.compare(model_pred == y_true, baseline_pred == y_true)
print(stats.format_report(comparison))

In [ ]:
table = np.array(comparison["table"])
fig, ax = plt.subplots(figsize=(5, 4))
ax.imshow(table, cmap="Purples")
ax.set_xticks([0, 1], ["baseline wrong", "baseline right"])
ax.set_yticks([0, 1], ["model wrong", "model right"])
ax.set_title("Paired outcomes")
for i in range(2):
    for j in range(2):
        ax.text(j, i, table[i, j], ha="center", va="center", fontsize=13)
plt.tight_layout()
plt.show()

The two off diagonal cells are the only ones the test uses.

The confidence interval on the accuracy difference gives the size of the effect, which the
p value alone does not.

## Full benchmark

`ocular benchmark` runs everything above, writes `benchmark.json` and the confusion matrix
figures to `artifacts/`, and prints a summary table.

In [ ]:
results = run(MANIFEST, ARTIFACTS / "model.pt", RAW, ARTIFACTS)
print(format_summary(results))

## Scope

The model classifies segments. It does not remove artifacts from a recording. Integration
into a preprocessing pipeline, and validation that the cleaned data remains suitable for
analysis, are outside the scope of this work.

Blinks in this dataset are cued, so they are deliberate and well separated. Spontaneous
blinks during a task are smaller and overlap with ongoing activity, and performance on
those is not measured here.